# SAPC Pharmacy Scraper

Scrapes the SAPC registry for active pharmacies in Gauteng and KwaZulu-Natal.

Uses a trie search over all 3-character prefixes to capture the full registry, then immediately fetches detail pages for Active records to retrieve address, phone, owner, and responsible pharmacist fields.

Key behaviors confirmed from the site:
- Active pharmacies return valid detail pages with full address data
- Inactive and Erased records return server errors on detail pages; table-level data only is kept
- Detail pages require the same session that performed the search

Estimated runtime is 1 to 2 hours for both provinces. The scrape is resumable from checkpoint if interrupted.

Output: `sapc_raw.csv`

In [ ]:
pip install requests beautifulsoup4 lxml pandas

## Imports and configuration

In [ ]:
import time, string, re, os, sys
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL   = 'https://interns.pharma.mm3.co.za'
SEARCH_URL = f'{BASE_URL}/SearchRegister'
DETAIL_URL = f'{BASE_URL}/SearchRegister/OrganisationSearchDetail'

PROVINCES = {'Gauteng': '3', 'KwaZulu-Natal': '4'}

OUTPUT_FILE     = 'sapc_raw.csv'
CHECKPOINT_FILE = 'sapc_checkpoint.csv'

SEARCH_DELAY   = 0.1    # seconds between search POSTs
DETAIL_WORKERS = 10     # parallel detail fetches per search batch
MIN_CHARS      = 3      # server-enforced search minimum
RESULT_CAP     = 100    # if a search returns >= this with no next page, expand the prefix
CHARS          = string.ascii_uppercase

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Gecko/20100101 Firefox/149.0',
    'Origin':  BASE_URL,
    'Referer': SEARCH_URL,
})

print(f'Config ready | Python {sys.version.split()[0]}')

## Helper functions

`warmup_and_token` primes the session and retrieves the CSRF token required for search POSTs.

`parse_table` extracts rows from the search results table and detects pagination.

`fetch_detail` retrieves the full detail page for a single Active pharmacy.

`fetch_details_parallel` runs detail fetches concurrently for a batch of Active rows.

In [ ]:
DETAIL_FIELD_MAP = {
    'account number':         'y_number',
    'pharmacy name':          'pharmacy_name',
    'name':                   'pharmacy_name',
    'license number':         'licence_number',
    'licence number':         'licence_number',
    'doh conditions':         'doh_conditions',
    'registration date':      'registration_date',
    'status':                 'status',
    'owner':                  'owner',
    'responsible pharmacist': 'responsible_pharmacist',
    'tutor(s)':               'tutors',
    'inspection':             'inspection',
    'province':               'province',
    'city':                   'city',
    'street address':         'street_address',
    'tel':                    'telephone',
    'provision':              'provision',
}


def warmup_and_token():
    resp = session.get(SEARCH_URL, timeout=15)
    soup = BeautifulSoup(resp.text, 'lxml')
    inp  = soup.find('input', {'name': '__RequestVerificationToken'})
    return inp['value'] if inp else None


def post_search(province_id, text, token):
    fields = {
        'OnlineSearchTypeId':         '2',
        'ProvinceId':                 province_id,
        'SearchText':                 text,
        '__RequestVerificationToken': token or '',
    }
    try:
        return session.post(SEARCH_URL,
                            files={k: (None, v) for k, v in fields.items()},
                            timeout=20)
    except requests.RequestException:
        return None


def parse_table(html):
    # Returns (list_of_row_dicts, has_next_page)
    soup  = BeautifulSoup(html, 'lxml')
    table = soup.find('table')
    if not table:
        return [], False
    all_tr = table.find_all('tr')
    if len(all_tr) < 2:
        return [], False

    headers = [th.get_text(strip=True) for th in all_tr[0].find_all(['th', 'td'])]
    pattern = re.compile(r'OrganisationSearchDetail/(\d+)', re.I)
    rows = []
    for tr in all_tr[1:]:
        cells = tr.find_all('td')
        if not cells:
            continue
        row = {headers[i]: cells[i].get_text(separator=' ', strip=True)
               for i in range(min(len(headers), len(cells)))}
        row['detail_id'] = None
        for a in tr.find_all('a', href=pattern):
            m = pattern.search(a['href'])
            if m:
                row['detail_id'] = m.group(1)
                break
        rows.append(row)

    has_next = False
    pager = soup.find(class_=re.compile(r'pager|pagination', re.I))
    if pager and pager.find('a', string=re.compile(r'next|>', re.I)):
        has_next = True
    return rows, has_next


def next_page_url(html):
    soup  = BeautifulSoup(html, 'lxml')
    pager = soup.find(class_=re.compile(r'pager|pagination', re.I))
    if not pager:
        return None
    link = pager.find('a', string=re.compile(r'next|>', re.I))
    if link and link.get('href'):
        href = link['href']
        return href if href.startswith('http') else BASE_URL + href
    return None


def fetch_detail(detail_id):
    # Must be called immediately after the search that returned this ID (same session required)
    # Returns {} for Inactive/Erased records — server returns an error page for those
    try:
        resp = session.get(f'{DETAIL_URL}/{detail_id}', timeout=15)
    except requests.RequestException:
        return {}
    if resp.status_code != 200 or 'An error occurred' in resp.text:
        return {}
    soup   = BeautifulSoup(resp.text, 'lxml')
    record = {}
    for tr in soup.find_all('tr'):
        th = tr.find('th')
        td = tr.find('td')
        if not th or not td:
            continue
        label = th.get_text(strip=True).lower()
        value = td.get_text(separator=' ', strip=True)
        col   = DETAIL_FIELD_MAP.get(label, re.sub(r'\s+', '_', label))
        if col not in record:
            record[col] = value
    return record


def fetch_details_parallel(active_rows):
    results = []
    ids     = [r for r in active_rows if r.get('detail_id')]
    if not ids:
        return active_rows
    with ThreadPoolExecutor(max_workers=DETAIL_WORKERS) as ex:
        future_to_row = {ex.submit(fetch_detail, r['detail_id']): r for r in ids}
        for future in as_completed(future_to_row):
            row    = future_to_row[future]
            detail = future.result()
            merged = dict(row)
            merged.update(detail)
            results.append(merged)
    no_id = [r for r in active_rows if not r.get('detail_id')]
    return results + no_id


print('Helpers ready.')

## Calibration

Confirms the full pipeline works — search, table parse, parallel detail fetch — before running the full scrape. Run this before the scrape cell.

In [ ]:
token = warmup_and_token()
print(f'Token: {token[:30]}...' if token else 'No token found')

# Test with 'DIS' to catch Dischem and similar chains
resp = post_search('3', 'DIS', token)
rows, has_next = parse_table(resp.text)
print(f'Search "DIS" Gauteng: {len(rows)} rows | has_next={has_next}')

if rows:
    active = [r for r in rows if r.get('Status', '').strip().lower() == 'active']
    print(f'Active: {len(active)} | Other: {len(rows) - len(active)}')
    if active:
        t0       = time.time()
        detailed = fetch_details_parallel(active[:5])
        elapsed  = time.time() - t0
        print(f'Parallel detail fetch (5 records): {elapsed:.1f}s')
        for r in detailed[:3]:
            print(f'  {r.get("Account #", "?"):>8} | {r.get("pharmacy_name", r.get("Name", "?"))[:30]}')
            print(f'           Address: {r.get("street_address", "[not fetched]")}')
else:
    print('No rows returned — inspect debug_search.html')
    with open('debug_search.html', 'w', encoding='utf-8') as f:
        f.write(resp.text)

## Full scrape

Iterates every 3-character prefix for both provinces. For each prefix it paginates fully, then parallel-fetches detail pages for Active records only. Inactive and Erased rows are kept with table data only.

Checkpoints every 100 new records. Resume by re-running this cell — already-seen Y-numbers are skipped.

In [ ]:
records    = []
seen_ynums = set()

# Resume from checkpoint if available
if os.path.exists(CHECKPOINT_FILE):
    chk = pd.read_csv(CHECKPOINT_FILE, dtype=str)
    records = chk.to_dict('records')
    for r in records:
        y = (r.get('y_number') or r.get('Account #') or '').strip()
        if y:
            seen_ynums.add(y)
    print(f'Resumed: {len(records)} records, {len(seen_ynums)} Y-numbers seen')
else:
    print('Starting fresh.')


def scrape_prefix(province_id, prefix, token):
    resp = post_search(province_id, prefix, token)
    if not resp or resp.status_code != 200:
        return [], 0, False

    all_rows = []
    html     = resp.text
    had_next = False

    while True:
        page_rows, has_next = parse_table(html)
        all_rows.extend(page_rows)
        if not has_next:
            break
        had_next = True
        nxt = next_page_url(html)
        if not nxt:
            break
        r = session.get(nxt, timeout=20)
        if r.status_code != 200:
            break
        html = r.text
        time.sleep(0.2)

    raw_count   = len(all_rows)
    active_rows = [r for r in all_rows
                   if r.get('Status', '').strip().lower() == 'active'
                   and r.get('detail_id')]

    complete     = fetch_details_parallel(active_rows) if active_rows else []
    inactive_rows = [r for r in all_rows if r.get('Status', '').strip().lower() != 'active']
    complete.extend(inactive_rows)

    return complete, raw_count, had_next


def run_province(province_label, province_id):
    global records, seen_ynums
    found  = 0
    reqs   = 0
    exp    = 0
    token  = warmup_and_token()
    queue  = deque(list(CHARS))

    while queue:
        prefix = queue.popleft()

        if len(prefix) < MIN_CHARS:
            for c in CHARS:
                queue.append(prefix + c)
            continue

        if reqs > 0 and reqs % 20 == 0:
            token = warmup_and_token()
            time.sleep(1)

        complete, raw_count, had_next = scrape_prefix(province_id, prefix, token)
        reqs += 1

        # Expand prefix if server truncated results (hit cap without pagination)
        if raw_count >= RESULT_CAP and not had_next:
            exp += 1
            for c in CHARS:
                queue.append(prefix + c)

        new = 0
        for row in complete:
            y = (row.get('y_number') or row.get('Account #') or '').strip()
            if not y or y in seen_ynums:
                continue
            seen_ynums.add(y)
            row['province_label'] = province_label
            records.append(row)
            new += 1
            found += 1

        if new > 0:
            print(f'  [{prefix}] {raw_count} rows | {new} new | total: {len(records)} | reqs: {reqs}')

        if found > 0 and found % 100 == 0:
            pd.DataFrame(records).to_csv(CHECKPOINT_FILE, index=False)
            print(f'  Checkpoint: {len(records)} records')

        time.sleep(SEARCH_DELAY if raw_count > 0 else 0.1)

    pd.DataFrame(records).to_csv(CHECKPOINT_FILE, index=False)
    print(f'\n  {province_label}: {found} pharmacies | {reqs} requests | {exp} expansions')


for label, pid in PROVINCES.items():
    print(f'\n{"="*50}\n  {label}\n{"="*50}')
    run_province(label, pid)
    time.sleep(5)

print(f'\nComplete: {len(records)} total records')

## Clean and export

Normalises column names, deduplicates on Y-number, orders columns, and saves to `sapc_raw.csv`.

In [ ]:
df = pd.DataFrame(records)

# Remove duplicate column names that arise when detail data overlaps table data
df = df.loc[:, ~df.columns.duplicated()]

# Normalise column names to snake_case
df.columns = [re.sub(r'[\s#]+', '_', str(c).strip().lower()).strip('_') for c in df.columns]

df.rename(columns={
    'name':     'pharmacy_name',
    'account_': 'y_number',
    'doh_':     'doh_number',
}, inplace=True)

# Deduplicate on Y-number
y_col = next((c for c in df.columns if c in ('y_number', 'account_')), None)
if y_col:
    before = len(df)
    df.drop_duplicates(subset=[y_col], inplace=True)
    print(f'Deduped on {y_col}: {before} -> {len(df)}')

preferred = [
    'y_number', 'pharmacy_name', 'licence_number', 'status', 'category',
    'registration_date', 'owner', 'responsible_pharmacist', 'inspection',
    'province', 'city', 'street_address', 'telephone',
    'doh_number', 'doh_conditions', 'province_label', 'detail_id',
]
ordered = [c for c in preferred if c in df.columns]
rest    = [c for c in df.columns if c not in preferred]
df      = df[ordered + rest]

df.to_csv(OUTPUT_FILE, index=False)
print(f'{len(df)} records saved to {OUTPUT_FILE}')
print()

if 'status' in df.columns:
    print('Status breakdown:')
    print(df['status'].value_counts().to_string())

if 'province_label' in df.columns:
    print('\nProvince breakdown:')
    print(df['province_label'].value_counts().to_string())

if 'street_address' in df.columns and 'status' in df.columns:
    active    = df[df['status'].str.lower() == 'active']
    with_addr = active['street_address'].notna() & (active['street_address'].str.strip() != '')
    print(f'\nActive pharmacies with address: {with_addr.sum()} / {len(active)}')

display(df[df['status'].str.lower() == 'active'].head(5))